# Nexar event captions v2: Qwen2.5-VL-72B (NF4), 32 frames with a dashcam prompt (§29.3)

Rule: `core/docs/final/KIM_RUN.md` **§29.3** (model Qwen2.5-VL-32B-Instruct in 4-bit NF4; dashcam prompt that opens with `Accident:` / `Near-collision:` / `Dangerous maneuver:`; prompt tuned on a 10-video dev set only (at most 2 more rounds after `6a24b2d9`; freeze iff dev mean >= 4.0 and `type_ok` >= 8/10); QC on 30 **fresh** videos, LaGoVAD Tab. B 1-5, pass iff mean >= 4.0), plus §29.2's window (16 frames over `[t_event − 2.0 s, t_event + 1.4 s]`). **A data artifact, not an arm**: the captions feed §30's arms only if QC passes. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **Runtime: an 80 GB GPU** (A100-80GB / H100; 72B in NF4 is ~40 GB).

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | mount, paths, deps (+ bitsandbytes), preflight | no | 3-4 min |
| 4 | pytest `test_nexar_caption.py` — **raises if red** | no | < 1 min |
| 5 | pin the model revision once (`revision.json`) | no | seconds |
| 6 | **dev set: 10 videos → paste `dev/<tag>/qc_sheet.md` to Claude and STOP.** Claude may change the prompt; re-upload and re-run 1-6 | yes | ~25 min first time (download ~145 GB to VM disk) |
| 7 | all 451 videos with the frozen prompt (resumable) + manifest | yes | ~4-6 h (700 x 392 frames, 2,800 visual tokens) |
| 8 | QC: 30 fresh seeded contact sheets + `qc/qc_ratings.csv` | no | 2-3 min |
| 9 | **you** fill `score` 1-5 (+ optional `type_ok` / `roles_ok` = y/n), then `qc-score` | no | your time |

Thesis and Thesis-V2 are only read; everything new goes to `Thesis-Final/outputs/nexar_captions_v2/` (§29.1's `nexar_captions/` is untouched).


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import csv
import json
import os
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data/Nexar/raw
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: the N0 census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'
os.environ['HF_HOME'] = '/content/hf'             # model weights on VM disk, not Drive FUSE

REPO = Path(f'{DRIVE_F}/kat-vad')
assert (REPO / 'core/tools/nexar_caption.py').is_file(), 'this upload has no nexar_caption -- re-upload kat-vad/'
assert '### 29.3' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §29.3 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
MODEL  = constants.NEXAR_CAPTION_MODEL
QUANT  = constants.NEXAR_CAPTION_QUANT       # 'nf4' (40 GB A100); 'none' needs ~70 GB
RAW    = constants.DATA_ROOT / constants.NEXAR_DATASET / 'raw'
CENSUS = Path(DRIVE_V2) / 'outputs' / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
OUT    = constants.OUTPUT_ROOT / constants.NEXAR_CAPTION_OUTPUT_DIRNAME
OUT.mkdir(parents=True, exist_ok=True)
for name, p in (('raw Nexar', RAW / constants.NEXAR_TRAIN_DIR), ('census', CENSUS)):
    assert p.exists(), f'missing {name}: {p} -- paste to Claude'
print(f'model {MODEL} ({QUANT}) | raw {RAW} | out {OUT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av "bitsandbytes>=0.45"
python -c "import av, bitsandbytes, torch, transformers; print('deps OK', transformers.__version__, 'bnb', bitsandbytes.__version__, 'torch', torch.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6-7 need an A100'
# §29.3 round 7: the 72B weights are ~145 GB on disk; drop the 32B cache first
rm -rf /content/hf/hub/models--Qwen--Qwen2.5-VL-32B-Instruct
df -h /content | tail -1
python -c "import shutil; f = shutil.disk_usage('/content').free / 2**30; print(f'free {f:.0f} GB'); assert f > 160, 'need > 160 GB free on /content for the 72B weights'"


In [ ]:
import torch  # noqa: E402

from core.tools import nexar_caption as nc  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    '''Stream a child's output here; raise with its tail on failure.'''
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


census = json.loads(CENSUS.read_text())
common = ['--raw-root', RAW, '--census', CENSUS, '--out-dir', OUT]
GEN = ['--quant', QUANT]
QC = OUT / nc.QC_DIRNAME
POS = nc.train_positives(census)                               # CP1: raises on a census issue
assert len(POS) == 451, len(POS)
DEV = nc.dev_sample(POS)
assert not set(DEV) & set(nc.qc_sample(POS)), 'dev and QC overlap'
missing = [v for v in POS if not (RAW / census['videos'][v]['path']).is_file()]
assert not missing, f'{len(missing)} positive mp4s missing under {RAW}: {missing[:5]}'
spans = [nc.event_frame_indices(census['videos'][v])[1] for v in POS]   # CP2: raises if a span is too short
print(f'preflight OK: {len(POS)} train positives, every mp4 present | span {min(e - s for s, e in spans):.2f}-'
      f'{max(e - s for s, e in spans):.2f} s | GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

## 4. Tests on the caption tool (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_nexar_caption.py'])

## 5. Pin the model revision once (C4)

The first run asks the Hub for the current commit of the model and writes it to `revision.json`; every later run (and the resume after a disconnect) reuses it, so all 451 captions come from one revision.

In [ ]:
from huggingface_hub import HfApi  # noqa: E402

REV_FILE = OUT / f"revision_{MODEL.split('/')[-1]}.json"   # one pin per model
if not REV_FILE.exists():
    REV_FILE.write_text(json.dumps({'model': MODEL, 'revision': HfApi().model_info(MODEL).sha}, indent=2))
REV = json.loads(REV_FILE.read_text())
assert REV['model'] == MODEL, REV
REVISION = REV['revision']
print(f'{MODEL} @ {REVISION}')

## 6. Dev set: 10 videos — **paste the printed sheet to Claude and STOP**

These 10 videos (seed 2027) are the only ones the prompt may be tuned on; they are never rated in QC. Each prompt version gets its own folder `dev/<prompt sha1[:8]>_<model>_<frames>f/`. If Claude changes the prompt, re-upload `kat-vad/` and re-run steps 1-6; go to step 7 only when Claude says the prompt is frozen.


In [ ]:
from IPython.display import Image, Markdown, display  # noqa: E402

assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'needs a bf16 GPU (A100) -- Runtime > Change runtime type'
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert gb >= 70, f'{gb:.0f} GB GPU: the 72B model in NF4 needs an 80 GB GPU (§29.3 round 7)'
print(f'GPU {torch.cuda.get_device_name(0)} {gb:.0f} GB')
run([sys.executable, '-m', 'core.tools.nexar_caption', 'run', *common, '--revision', REVISION, *GEN, '--subset', 'dev'])
DEV_DIR = OUT / 'dev' / nc.dev_dirname(MODEL, constants.NEXAR_CAPTION_PROMPT)
print((DEV_DIR / nc.QC_SHEET_FILENAME).read_text())      # <- paste this to Claude
for png in sorted(DEV_DIR.glob('*.png')):
    print(png.stem)
    display(Image(filename=str(png), width=900))


## 7. All 451 videos + manifest (resumable) — **only after Claude froze the prompt**

**A Colab disconnect is harmless**: re-run steps 1-5 and this cell; finished videos are skipped.

In [ ]:
t0 = time.time()
run([sys.executable, '-m', 'core.tools.nexar_caption', 'run', *common, '--revision', REVISION, *GEN])
m = json.loads((OUT / nc.MANIFEST_FILENAME).read_text())
assert m['videos'] == len(POS) and m['revision'] == REVISION, (m['videos'], m['revision'])
print(f"{m['videos']} captions in {(time.time() - t0) / 60:.0f} min | summary {json.dumps(m['summary'])}")
drive.flush_and_unmount()
drive.mount('/content/drive')

## 8. QC sheets (§29.3 CP6'': 30 videos, seed 2026, outside the dev set and the §29.1 draw)

Each PNG = the 16 frames the model saw, in reading order. Rate them in step 9.


In [ ]:
run([sys.executable, '-m', 'core.tools.nexar_caption', 'qc', *common])
display(Markdown((QC / nc.QC_SHEET_FILENAME).read_text()))
for png in sorted(QC.glob('*.png')):
    print(png.stem)
    display(Image(filename=str(png), width=900))
print(f'Now open {QC / nc.QC_RATINGS_FILENAME} on Drive, fill the score column with an integer 1-5 (Tab. B above) for all 30 rows (optional: type_ok / roles_ok = y or n), save, then run step 9.')

## 9. QC score — **after you filled `qc_ratings.csv`; paste the printed line to Claude**

In [ ]:
run([sys.executable, '-m', 'core.tools.nexar_caption', 'qc-score', '--out-dir', OUT])
print(json.loads((QC / nc.QC_SUMMARY_FILENAME).read_text()))
drive.flush_and_unmount()          # C10: every pending Drive write lands before the VM goes